In [1]:
import numpy as np
import pandas as pd

In [2]:
import warnings, uuid
from collections import defaultdict
from datetime import datetime
warnings.filterwarnings("ignore")


In [3]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD

In [4]:
SKILL_CATALOG = {
    # Tech
    "Python Programming":       {"domain": "Tech",     "demand": "Very High"},
    "Web Development":          {"domain": "Tech",     "demand": "Very High"},
    "Machine Learning":         {"domain": "Tech",     "demand": "Very High"},
    "Data Analysis":            {"domain": "Tech",     "demand": "High"},
    "Mobile App Development":   {"domain": "Tech",     "demand": "High"},
    "Database & SQL":           {"domain": "Tech",     "demand": "High"},
    "Cybersecurity":            {"domain": "Tech",     "demand": "High"},
    "Cloud Computing":          {"domain": "Tech",     "demand": "High"},
    "UI/UX Design":             {"domain": "Tech",     "demand": "High"},
    "Git & Version Control":    {"domain": "Tech",     "demand": "Moderate"},

    # Science & Maths
    "Statistics":               {"domain": "Science",  "demand": "High"},
    "Mathematics":              {"domain": "Science",  "demand": "High"},
    "Physics":                  {"domain": "Science",  "demand": "Moderate"},
    "Chemistry":                {"domain": "Science",  "demand": "Moderate"},
    "Biology":                  {"domain": "Science",  "demand": "Moderate"},
    "Research Methods":         {"domain": "Science",  "demand": "High"},

    # Business
    "Business Communication":   {"domain": "Business", "demand": "High"},
    "Marketing & SEO":          {"domain": "Business", "demand": "High"},
    "Finance & Accounting":     {"domain": "Business", "demand": "High"},
    "Entrepreneurship":         {"domain": "Business", "demand": "High"},
    "Project Management":       {"domain": "Business", "demand": "High"},
    "Public Speaking":          {"domain": "Business", "demand": "High"},
    "Negotiation":              {"domain": "Business", "demand": "Moderate"},

    # Creative
    "Graphic Design":           {"domain": "Creative", "demand": "Moderate"},
    "Video Editing":            {"domain": "Creative", "demand": "High"},
    "Content Writing":          {"domain": "Creative", "demand": "High"},
    "Photography":              {"domain": "Creative", "demand": "Moderate"},
    "Music Production":         {"domain": "Creative", "demand": "Moderate"},

    # Language
    "English Communication":    {"domain": "Language", "demand": "Very High"},
    "Hindi":                    {"domain": "Language", "demand": "Moderate"},
    "French":                   {"domain": "Language", "demand": "Moderate"},
    "German":                   {"domain": "Language", "demand": "Moderate"},

    # Soft Skills
    "Leadership":               {"domain": "Soft",     "demand": "High"},
    "Critical Thinking":        {"domain": "Soft",     "demand": "High"},
    "Time Management":          {"domain": "Soft",     "demand": "Moderate"},
}


In [5]:
ALL_SKILLS = list(SKILL_CATALOG.keys())

CAREER_SKILL_MAP = {
    "Software Engineer":             ["Python Programming", "Web Development", "Database & SQL",
                                      "Git & Version Control", "Critical Thinking"],
    "Data Scientist":                ["Python Programming", "Machine Learning", "Statistics",
                                      "Data Analysis", "Research Methods"],
    "Doctor / Medical Professional": ["Biology", "Chemistry", "Research Methods",
                                      "English Communication", "Critical Thinking"],
    "Civil Engineer":                ["Mathematics", "Physics", "Project Management",
                                      "Research Methods"],
    "Graphic Designer / UX Designer":["UI/UX Design", "Graphic Design", "Content Writing",
                                      "English Communication"],
    "Business Analyst / MBA":        ["Finance & Accounting", "Business Communication",
                                      "Statistics", "Project Management", "Public Speaking"],
    "Lawyer / Legal Professional":   ["English Communication", "Public Speaking",
                                      "Critical Thinking", "Research Methods", "Negotiation"],
    "Psychologist / Counselor":      ["English Communication", "Research Methods",
                                      "Biology", "Leadership"],
    "Teacher / Educator":            ["Public Speaking", "English Communication",
                                      "Leadership", "Content Writing"],
    "Entrepreneur":                  ["Entrepreneurship", "Marketing & SEO",
                                      "Finance & Accounting", "Leadership", "Negotiation"],
    "Mechanical Engineer":           ["Mathematics", "Physics", "Project Management",
                                      "Critical Thinking"],
    "Journalist / Content Creator":  ["Content Writing", "English Communication",
                                      "Video Editing", "Research Methods", "Public Speaking"],
}


In [6]:
class StudentProfile:

    def __init__(
        self,
        name: str,
        skills_can_teach: dict,
        skills_want_learn: list,
        career_goal: str = None,
        student_id: str = None,
    ):
        self.student_id       = student_id or str(uuid.uuid4())[:6].upper()
        self.name             = name
        self.skills_can_teach  = skills_can_teach  
        self.skills_want_learn = list(skills_want_learn)
        self.career_goal       = career_goal
        self.joined_on         = datetime.now().strftime("%Y-%m-%d")
        self.rating            = 5.0
        self.total_exchanges   = 0

    def career_skill_gap(self) -> list:
        if not self.career_goal or self.career_goal not in CAREER_SKILL_MAP:
            return []
        required = set(CAREER_SKILL_MAP[self.career_goal])
        owned = {s for s, lvl in self.skills_can_teach.items() if lvl >= 2}
        return sorted(required - owned)

    def teach_vector(self) -> np.ndarray:
        return np.array(
            [self.skills_can_teach.get(s, 0) for s in ALL_SKILLS],
            dtype=float
        )

    def want_vector(self) -> np.ndarray:
        return np.array(
            [1.0 if s in self.skills_want_learn else 0.0 for s in ALL_SKILLS]
        )

    def __repr__(self):
        n_teach = len(self.skills_can_teach)
        n_want  = len(self.skills_want_learn)
        return (f"<Student [{self.student_id}] {self.name} | "
                f"Teaches: {n_teach} | Wants: {n_want} | Goal: {self.career_goal}>")


In [7]:
class SkillExchangePlatform:
    def __init__(self, name: str = "SkillBridge"):
        self.name     = name
        self.students : dict[str, StudentProfile] = {}
        self.exchanges: list[dict] = []  

    def register(self, student: StudentProfile) -> str:
        self.students[student.student_id] = student
        print(f" Registered  →  {student.name}  [{student.student_id}]")
        return student.student_id

    def get(self, sid: str) -> StudentProfile | None:
        return self.students.get(sid)

    def all_students(self) -> list[StudentProfile]:
        return list(self.students.values())

    def log_exchange(
        self,
        student_a_id: str,
        student_b_id: str,
        a_teaches: str,  
        b_teaches: str,    
        rating_a: float = 5.0,
        rating_b: float = 5.0,
    ) -> str:
        eid = str(uuid.uuid4())[:8].upper()
        record = {
            "exchange_id": eid,
            "student_a":   student_a_id,
            "student_b":   student_b_id,
            "a_teaches_b": a_teaches,
            "b_teaches_a": b_teaches,
            "rating_a":    rating_a,
            "rating_b":    rating_b,
            "date":        datetime.now().strftime("%Y-%m-%d"),
        }
        self.exchanges.append(record)


        for sid, rating in [(student_a_id, rating_a), (student_b_id, rating_b)]:
            s = self.students[sid]
            s.rating = round(
                (s.rating * s.total_exchanges + rating) / (s.total_exchanges + 1), 2
            )
            s.total_exchanges += 1

        a_name = self.students[student_a_id].name
        b_name = self.students[student_b_id].name
        print(f" Exchange [{eid}]: {a_name} ↔ {b_name}  "
              f"({a_teaches} ↔ {b_teaches})")
        return eid

    def platform_summary(self):
        print(f"\n {self.name} Platform Summary ")
        print(f" Registered students : {len(self.students)}")
        print(f" Completed exchanges : {len(self.exchanges)}")
        total_skills_taught = sum(
            len(s.skills_can_teach) for s in self.students.values()
        )
        print(f"Total skills listed : {total_skills_taught}")



In [8]:
class BarterMatchEngine:

    WEIGHTS = {
        "mutual_match":   0.40,
        "career_gap":     0.30,
        "proficiency":    0.15,
        "want_similarity":0.10,
        "rating":         0.05,
    }

    def __init__(self, platform: SkillExchangePlatform):
        self.platform = platform


    def _mutual_match(self, a: StudentProfile, b: StudentProfile) -> tuple[float, list, list]:
        
        a_teaches = set(a.skills_can_teach.keys())
        b_teaches = set(b.skills_can_teach.keys())
        a_wants   = set(a.skills_want_learn)
        b_wants   = set(b.skills_want_learn)

        a_gives_b = sorted(a_teaches & b_wants)   
        b_gives_a = sorted(b_teaches & a_wants)   

        if not a_gives_b or not b_gives_a:
            one_side = len(a_gives_b) + len(b_gives_a)
            score = 0.25 * min(one_side / max(len(a_wants) + len(b_wants), 1), 1.0)
            return score, a_gives_b, b_gives_a
        score_ab = len(a_gives_b) / max(len(b_wants), 1)
        score_ba = len(b_gives_a) / max(len(a_wants), 1)
        score    = (score_ab + score_ba) / 2
        return score, a_gives_b, b_gives_a

    def _career_gap_score(self, a: StudentProfile, b: StudentProfile) -> float:
       
        gap_a = set(a.career_skill_gap())
        gap_b = set(b.career_skill_gap())

        fill_a = len(set(b.skills_can_teach.keys()) & gap_a) / max(len(gap_a), 1) if gap_a else 0.5
        fill_b = len(set(a.skills_can_teach.keys()) & gap_b) / max(len(gap_b), 1) if gap_b else 0.5
        return (fill_a + fill_b) / 2

    def _proficiency_score(self, a: StudentProfile, b: StudentProfile,
                           a_gives: list, b_gives: list) -> float:
       
        def avg_prof(giver: StudentProfile, skills: list) -> float:
            if not skills:
                return 0.0
            levels = [giver.skills_can_teach.get(s, 0) for s in skills]
            return np.mean(levels) / 5.0   # normalize to 0-1

        score_a = avg_prof(a, a_gives)
        score_b = avg_prof(b, b_gives)
        return (score_a + score_b) / 2

    def _want_similarity(self, a: StudentProfile, b: StudentProfile) -> float:
        
        va = a.want_vector().reshape(1, -1)
        vb = b.want_vector().reshape(1, -1)
        if va.sum() == 0 or vb.sum() == 0:
            return 0.0
        return float(cosine_similarity(va, vb)[0][0])

    def _rating_score(self, a: StudentProfile, b: StudentProfile) -> float:
       
        return ((a.rating + b.rating) / 2) / 5.0


    def find_matches(self,student_id: str,top_n: int = 5,only_mutual: bool = False,) -> list[dict]:
       
        student = self.platform.get(student_id)
        if not student:
            print(f" Student {student_id} not found.")
            return []

        W = self.WEIGHTS
        results = []

        for other in self.platform.all_students():
            if other.student_id == student_id:
                continue

            mutual, a_gives, b_gives = self._mutual_match(student, other)

            if only_mutual and (not a_gives or not b_gives):
                continue

            factors = {
                "mutual_match":    mutual,
                "career_gap":      self._career_gap_score(student, other),
                "proficiency":     self._proficiency_score(student, other, a_gives, b_gives),
                "want_similarity": self._want_similarity(student, other),
                "rating":          self._rating_score(student, other),
            }

            final = sum(W[k] * v for k, v in factors.items())

            results.append({
                "partner":         other,
                "final_score":     round(final, 4),
                "match_pct":       round(final * 100, 1),
                "is_mutual":       bool(a_gives and b_gives),
                "student_teaches": a_gives,     # what the queried student teaches partner
                "partner_teaches": b_gives,     # what partner teaches the queried student
                "factor_scores":   {k: round(v * 100, 1) for k, v in factors.items()},
            })

        results.sort(key=lambda x: x["final_score"], reverse=True)
        return results[:top_n]

    def build_match_matrix(self) -> pd.DataFrame:
    
        sids   = list(self.platform.students.keys())
        names  = [self.platform.get(s).name for s in sids]
        n      = len(sids)
        matrix = np.zeros((n, n))

        for i, sid_a in enumerate(sids):
            for j, sid_b in enumerate(sids):
                if i == j:
                    continue
                mutual, ag, bg = self._mutual_match(
                    self.platform.get(sid_a), self.platform.get(sid_b)
                )
                matrix[i, j] = round(mutual, 3)

        return pd.DataFrame(matrix, index=names, columns=names)


In [9]:
class SkillRecommender:
    def __init__(self, platform: SkillExchangePlatform):
        self.platform = platform

    def career_gap_recommendations(
        self, student_id: str, top_n: int = 5
    ) -> list[dict]:
        student  = self.platform.get(student_id)
        if not student:
            return []

        gap = student.career_skill_gap()
        if not student.career_goal:
            return []

        recs = []
        for skill in gap:
            info = SKILL_CATALOG.get(skill, {})
            # Count how many platform members can teach this skill
            available_teachers = [
                s for s in self.platform.all_students()
                if skill in s.skills_can_teach
                and s.student_id != student_id
                and s.skills_can_teach[skill] >= 2
            ]
            recs.append({
                "skill":       skill,
                "strategy":    "Career Gap",
                "reason":      f"Required for '{student.career_goal}' — not yet in your profile",
                "domain":      info.get("domain", "—"),
                "demand":      info.get("demand", "—"),
                "teachers_on_platform": len(available_teachers),
                "teacher_names": [t.name for t in available_teachers[:3]],
                "priority":    "High" if info.get("demand") in ["Very High","High"] else "Medium",
            })

       
        recs.sort(key=lambda x: -x["teachers_on_platform"])
        return recs[:top_n]


    def collaborative_recommendations(
        self, student_id: str, top_n: int = 5
    ) -> list[dict]:
        all_students = self.platform.all_students()
        if len(all_students) < 4:
            return []

        teach_matrix = np.array(
            [s.teach_vector() for s in all_students], dtype=float
        )

        n_comp = min(10, teach_matrix.shape[1] - 1, teach_matrix.shape[0] - 1)
        svd    = TruncatedSVD(n_components=n_comp, random_state=42)
        latent = svd.fit_transform(teach_matrix)

        idx = next(
            i for i, s in enumerate(all_students)
            if s.student_id == student_id
        )
        target_vec = latent[idx].reshape(1, -1)
        sims = cosine_similarity(target_vec, latent)[0]
        sims[idx] = -1 

        top_peers = np.argsort(sims)[-3:][::-1]

        student  = self.platform.get(student_id)
        owned    = set(student.skills_can_teach.keys())
        wanted   = set(student.skills_want_learn)

        skill_scores: dict[str, float] = defaultdict(float)
        for pi in top_peers:
            peer   = all_students[pi]
            weight = max(sims[pi], 0)
            for skill, level in peer.skills_can_teach.items():
                if skill not in owned:
                    bonus = 1.5 if skill in wanted else 1.0
                    skill_scores[skill] += weight * level * bonus

        recs = []
        for skill, score in sorted(
            skill_scores.items(), key=lambda x: -x[1]
        )[:top_n]:
            info     = SKILL_CATALOG.get(skill, {})
            teachers = [
                s for s in all_students
                if skill in s.skills_can_teach and s.student_id != student_id
            ]
            recs.append({
                "skill":       skill,
                "strategy":    "Collaborative Filter",
                "reason":      "Popular among students with similar skill profiles",
                "domain":      info.get("domain", "—"),
                "demand":      info.get("demand", "—"),
                "collab_score":round(score, 3),
                "teachers_on_platform": len(teachers),
                "teacher_names": [t.name for t in teachers[:3]],
                "priority":    " Trending",
            })
        return recs

    def recommend(self, student_id: str, top_n: int = 6) -> list[dict]:
        gap_recs    = self.career_gap_recommendations(student_id, top_n)
        collab_recs = self.collaborative_recommendations(student_id, top_n)

        seen, combined = set(), []
        for r in gap_recs + collab_recs:
            if r["skill"] not in seen:
                seen.add(r["skill"])
                combined.append(r)
        return combined[:top_n]



In [10]:
class PeerClusterer:
    def __init__(self, platform: SkillExchangePlatform, n_groups: int = 4):
        self.platform  = platform
        self.n_groups  = n_groups

    def cluster(self) -> dict[int, list[StudentProfile]]:
        students = self.platform.all_students()
        if len(students) < self.n_groups:
            print(" Too few students to cluster into groups.")
            return {}

        matrix = np.array([s.teach_vector() for s in students], dtype=float)
        k      = min(self.n_groups, len(students))
        km     = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = km.fit_predict(matrix)

        groups: dict[int, list] = defaultdict(list)
        for i, s in enumerate(students):
            s.group = int(labels[i])
            groups[s.group].append(s)

        print(f"\n   Peer Learning Groups  (k={k})")
        for gid, members in sorted(groups.items()):
            names   = ", ".join(m.name for m in members)
            domains = set()
            for m in members:
                for sk in m.skills_can_teach:
                    domains.add(SKILL_CATALOG.get(sk, {}).get("domain", ""))
            domains.discard("")
            print(f"  Group {gid+1} [{len(members)} students]: {names}")
            print(f"           Skill domains: {', '.join(sorted(domains))}\n")

        return dict(groups)


In [11]:
def print_match_report(student: StudentProfile, matches: list[dict]):
    print(f"  Barter Matches for : {student.name}  [{student.student_id}]")
    print(f"  Career Goal           : {student.career_goal or 'Not specified'}")
    print(f"  I can teach           : {', '.join(student.skills_can_teach) or '—'}")
    print(f"  I want to learn       : {', '.join(student.skills_want_learn) or '—'}")
    print(f"  Career skill gap      : {', '.join(student.career_skill_gap()) or 'None'}")
    print(f"{'-'*64}")

    if not matches:
        print("\n  No matches found. Try adding more students to the platform.\n")
        return

    for rank, m in enumerate(matches, 1):
        p   = m["partner"]
        tag = "Mutual Barter" if m["is_mutual"] else " One-sided"
        print(f"\n  #{rank}  {p.name}  [{p.student_id}]  —  {tag}")
        print(f"       Match Score     : {m['match_pct']}%")
        print(f"       Career Goal     : {p.career_goal or '—'}")
        print(f"       You teach them  : {', '.join(m['student_teaches']) or '—'}")
        print(f"       They teach you  : {', '.join(m['partner_teaches']) or '—'}")
        f = m["factor_scores"]
        print(f"       Scores          : Mutual={f['mutual_match']}%  "
              f"Career={f['career_gap']}%  "
              f"Proficiency={f['proficiency']}%")
        
    print()


In [12]:
def print_recommendation_report(student: StudentProfile, recs: list[dict]):
    print(f" Skill Recommendations for : {student.name}")
    print(f"  Career Goal                   : {student.career_goal or 'Not specified'}")

    if not recs:
        print("\n  No recommendations — your profile looks complete!\n")
        return

    for i, r in enumerate(recs, 1):
        teachers = ", ".join(r["teacher_names"]) if r["teacher_names"] else "None yet"
        print(f"\n  #{i}  {r['skill']}  {r['priority']}")
        print(f"       Strategy   : {r['strategy']}")
        print(f"       Reason     : {r['reason']}")
        print(f"       Domain     : {r['domain']}  |  Market Demand: {r['demand']}")
        print(f"       Who can teach you on this platform: {teachers}")
    print()


In [13]:
def create_demo_students() -> list[StudentProfile]:
    
    return [
        StudentProfile(
            name             = "Sameer",
            career_goal      = "Data Scientist",
            skills_can_teach = {"Python Programming": 4, "Mathematics": 4,
                                 "Statistics": 3, "Research Methods": 3},
            skills_want_learn= ["Machine Learning", "Data Analysis",
                                 "Public Speaking", "Content Writing"],
        ),
        StudentProfile(
            name             = "Arshad",
            career_goal      = "Software Engineer",
            skills_can_teach = {"Web Development": 4, "Database & SQL": 3,
                                 "Git & Version Control": 4, "UI/UX Design": 2},
            skills_want_learn= ["Python Programming", "Machine Learning",
                                 "Cloud Computing", "Critical Thinking"],
        ),
        StudentProfile(
            name             = "Rumaisa",
            career_goal      = "Graphic Designer / UX Designer",
            skills_can_teach = {"Graphic Design": 5, "UI/UX Design": 4,
                                 "Photography": 3, "Content Writing": 3},
            skills_want_learn= ["Web Development", "Video Editing",
                                 "Marketing & SEO", "Python Programming"],
        ),
        StudentProfile(
            name             = "Vethasri",
            career_goal      = "Entrepreneur",
            skills_can_teach = {"Entrepreneurship": 4, "Marketing & SEO": 4,
                                 "Public Speaking": 3, "Finance & Accounting": 3},
            skills_want_learn= ["Python Programming", "Data Analysis",
                                 "Web Development", "Leadership"],
        ),
        StudentProfile(
            name             = "Sharan",
            career_goal      = "Business Analyst / MBA",
            skills_can_teach = {"Finance & Accounting": 4, "Project Management": 4,
                                 "Business Communication": 4, "Statistics": 2},
            skills_want_learn= ["Python Programming", "Data Analysis",
                                 "Machine Learning", "Public Speaking"],
        ),
        StudentProfile(
            name             = "Reshma",
            career_goal      = "Journalist / Content Creator",
            skills_can_teach = {"Content Writing": 5, "Video Editing": 4,
                                 "English Communication": 4, "Photography": 3},
            skills_want_learn= ["Graphic Design", "Marketing & SEO",
                                 "Public Speaking", "UI/UX Design"],
        ),
        StudentProfile(
            name             = "Abhishek",
            career_goal      = "Software Engineer",
            skills_can_teach = {"Python Programming": 5, "Machine Learning": 3,
                                 "Database & SQL": 4, "Mathematics": 3},
            skills_want_learn= ["Web Development", "UI/UX Design",
                                 "Cloud Computing", "Public Speaking"],
        ),
        StudentProfile(
            name             = "Pooja",
            career_goal      = "Doctor / Medical Professional",
            skills_can_teach = {"Biology": 5, "Chemistry": 4,
                                 "Research Methods": 3, "English Communication": 3},
            skills_want_learn= ["Statistics", "Python Programming",
                                 "Data Analysis", "Public Speaking"],
        ),
        StudentProfile(
            name             = "Dharshana",
            career_goal      = "Teacher / Educator",
            skills_can_teach = {"Public Speaking": 5, "Leadership": 4,
                                 "English Communication": 4, "Mathematics": 3},
            skills_want_learn= ["Content Writing", "Video Editing",
                                 "Python Programming", "Project Management"],
        ),
        StudentProfile(
            name             = "Priya",
            career_goal      = "Psychologist / Counselor",
            skills_can_teach = {"English Communication": 4, "Research Methods": 3,
                                 "Leadership": 3, "Biology": 2},
            skills_want_learn= ["Statistics", "Python Programming",
                                 "Content Writing", "Public Speaking"],
        ),
        StudentProfile(
            name             = "Rasika",
            career_goal      = "Mechanical Engineer",
            skills_can_teach = {"Mathematics": 5, "Physics": 4,
                                 "Project Management": 3, "Critical Thinking": 3},
            skills_want_learn= ["Python Programming", "Data Analysis",
                                 "Machine Learning", "Cloud Computing"],
        ),
        StudentProfile(
            name             = "Hari",
            career_goal      = "Lawyer / Legal Professional",
            skills_can_teach = {"English Communication": 5, "Research Methods": 4,
                                 "Negotiation": 4, "Public Speaking": 3},
            skills_want_learn= ["Statistics", "Finance & Accounting",
                                 "Data Analysis", "Python Programming"],
        ),
    ]


In [14]:
def run_demo():
    print("     AI-Based Skill Exchange System — Barter Mode")

    platform = SkillExchangePlatform("SkillBridge")
    students = create_demo_students()

    print("Registering students\n")
    for s in students:
        platform.register(s)
    platform.platform_summary()

    engine      = BarterMatchEngine(platform)
    recommender = SkillRecommender(platform)
    clusterer   = PeerClusterer(platform, n_groups=4)

    clusterer.cluster()

    demo = students[0]   
    print(f"Finding barter matches for: {demo.name}\n")
    matches = engine.find_matches(demo.student_id, top_n=5)
    print_match_report(demo, matches)

    recs = recommender.recommend(demo.student_id, top_n=6)
    print_recommendation_report(demo, recs)

    if matches and matches[0]["is_mutual"]:
        best    = matches[0]
        partner = best["partner"]
        teach   = best["student_teaches"][0]
        receive = best["partner_teaches"][0]
        platform.log_exchange(
            demo.student_id, partner.student_id,
            a_teaches=teach, b_teaches=receive,
            rating_a=4.8, rating_b=4.9,
        )


    print("\nBuilding compatibility matrix (first 5 students)...")
    matrix = engine.build_match_matrix()
    print(matrix.iloc[:5, :5].round(2).to_string())

    
    print("\nSkill Exchange System complete!\n")
    return platform, engine, recommender



In [15]:
if __name__ == "__main__":
    platform, engine, recommender = run_demo()

     AI-Based Skill Exchange System — Barter Mode
Registering students

 Registered  →  Sameer  [B6626F]
 Registered  →  Arshad  [9F5FE2]
 Registered  →  Rumaisa  [38662F]
 Registered  →  Vethasri  [48C934]
 Registered  →  Sharan  [B6F37C]
 Registered  →  Reshma  [FA8774]
 Registered  →  Abhishek  [427432]
 Registered  →  Pooja  [D1DF81]
 Registered  →  Dharshana  [1A1E15]
 Registered  →  Priya  [13FF6B]
 Registered  →  Rasika  [0516EC]
 Registered  →  Hari  [92E472]

 SkillBridge Platform Summary 
 Registered students : 12
 Completed exchanges : 0
Total skills listed : 48

   Peer Learning Groups  (k=4)
  Group 1 [3 students]: Arshad, Vethasri, Sharan
           Skill domains: Business, Science, Tech

  Group 2 [4 students]: Pooja, Dharshana, Priya, Hari
           Skill domains: Business, Language, Science, Soft

  Group 3 [3 students]: Sameer, Abhishek, Rasika
           Skill domains: Business, Science, Soft, Tech

  Group 4 [2 students]: Rumaisa, Reshma
           Skill domains: C